In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import LearningRateScheduler




train_path  = '/kaggle/input/digit-recognizer/train.csv'
test_path = '/kaggle/input/digit-recognizer/test.csv'
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

In [ ]:
from tensorflow.keras.utils import to_categorical


Y_train = train["label"]
X_train = train.drop(labels = ["label"],axis = 1)
X_train = X_train / 255.0
X_test = test / 255.0
X_train = X_train.values.reshape(-1,28,28,1)
X_test = X_test.values.reshape(-1,28,28,1)
Y_train = to_categorical(Y_train, num_classes = 10)



In [ ]:
import matplotlib.pyplot as plt

# CREATE MORE IMAGES VIA DATA AUGMENTATION
datagen = ImageDataGenerator(
        rotation_range=4,  
        zoom_range = 0.05,  
        width_shift_range=0.05, 
        height_shift_range=0.05)



# PREVIEW AUGMENTED IMAGES
X_train3 = X_train[9,].reshape((1,28,28,1))
Y_train3 = Y_train[9,].reshape((1,10))
plt.figure(figsize=(15,4.5))
for i in range(30):  
    plt.subplot(3, 10, i+1)
    X_train2, Y_train2 = next(datagen.flow(X_train3, Y_train3))
    plt.imshow(X_train2[0].reshape((28,28)),cmap=plt.cm.binary)
    plt.axis('off')
    if i==9: X_train3 = X_train[11,].reshape((1,28,28,1))
    if i==19: X_train3 = X_train[18,].reshape((1,28,28,1))
plt.subplots_adjust(wspace=-0.1, hspace=-0.1)
plt.show()



In [ ]:


# 1. Configuration
augmentations_par_image = 3  # Combien d'images générées par image
img_rows, img_cols = 28, 28

# 2. Convertir les vecteurs 784 en images 28x28x1
X_train_reshaped = X_train.reshape((-1, img_rows, img_cols, 1))

# 3. Préparer la data augmentation
datagen = ImageDataGenerator(
    rotation_range= 7 , 
    zoom_range=0.05,
    width_shift_range=0.05,
    height_shift_range=0.05
)

# 4. Créer les nouveaux tableaux qui contiendront les images augmentées
X_augmented = []
Y_augmented = []

# 5. Boucle sur toutes les images
for i in range(X_train_reshaped.shape[0]):
    x = X_train_reshaped[i].reshape((1, img_rows, img_cols, 1))  # image source
    y = Y_train[i].reshape((1, 10))  # label one-hot

    # Générer `augmentations_par_image` nouvelles images à partir de celle-ci
    for _ in range(augmentations_par_image):
        x_aug, y_aug = next(datagen.flow(x, y, batch_size=1))
        X_augmented.append(x_aug[0])  # on enlève la 1ère dimension batch
        Y_augmented.append(y_aug[0])

# 6. Conversion en numpy arrays
X_augmented = np.array(X_augmented)
Y_augmented = np.array(Y_augmented)

# 7. Ré-aplatir les images vers format 784
X_augmented_flat = X_augmented.reshape((-1, 784))
X_train_flat = X_train.reshape((X_train.shape[0], -1))  # Devient (42000, 784)

# 8. Fusion avec les données d'origine (corrigé)
X_total = np.concatenate((X_train_flat, X_augmented_flat), axis=0)
Y_total = np.concatenate((Y_train, Y_augmented), axis=0)


# Affichage des tailles finales
print("X_total shape:", X_total.shape)
print("Y_total shape:", Y_total.shape)
  

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Création du modèle
model = Sequential()

# Input layer + première couche cachée
model.add(Dense(128, activation='relu', input_shape=(784,)))
model.add(Dropout(0.3))  # pour éviter l'overfitting

# Deuxième couche cachée
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))

# Couche de sortie
model.add(Dense(10, activation='softmax' ))  # 10 classes (0 à 9)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',  # car on a fait du one-hot
    metrics=['accuracy']
)


from sklearn.model_selection import train_test_split

# Split en 80% entraînement / 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_total, Y_total,           # données + labels
    test_size=0.1,          # 20% pour la validation
    random_state=1,        # pour rendre le split reproductible
    shuffle=True            # très important : mélange les données
)

model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=128,
    verbose=1
)

In [ ]:
num = 208
image = X_total[num].reshape(1, -1)
prediction = model.predict(image)       
print(prediction)
img = X_total[num]
label = Y_total[num]

# Reshape pour retrouver l'image 28x28
img = img.reshape(28,28)

# Afficher l'image
plt.imshow(img, cmap='gray')
plt.title(f"Label: {label}")
plt.axis('off')
plt.show()

In [ ]:
X_test = test / 255.0
print("X_test shape:", X_test.shape)

In [ ]:

predscomp = model.predict(X_test)
predscompclass = np.argmax(predscomp, axis=1)
submission = pd.DataFrame({
    "ImageId": np.arange(1, len(predscompclass) + 1),
    "Label": predscompclass
})

submission.to_csv("submission.csv", index=False)

In [ ]:
model.save("DigitRecognizerV2.h5")